In [ ]:
import joblib
import pandas as pd
import xgboost as xgb
import os

# usage: python inspect_model.py

def inspect_pickle(filename):
    print(f"\n{'='*60}")
    print(f"🔍 INSPECTING: {filename}")
    print(f"{'='*60}")

    if not os.path.exists(filename):
        print("❌ File not found.")
        return

    try:
        # 1. Load the binary file
        model = joblib.load(filename)
        print(f"✅ Type: {type(model).__name__}")

        # 2. Check if it is a valid XGBoost model
        if hasattr(model, 'get_params'):
            params = model.get_params()
            
            # 3. Print Key indicators of a "Dummy" vs "Real" model
            n_estimators = params.get('n_estimators', 'Unknown')
            max_depth = params.get('max_depth', 'Unknown')
            
            print(f"📊 Number of Trees (n_estimators): {n_estimators}")
            print(f"📉 Tree Depth (max_depth):        {max_depth}")
            
            # Real models have 100+ trees. Dummy models usually have 1.
            if n_estimators == 1 and max_depth == 1:
                print("\n⚠️  VERDICT: DUMMY MODEL DETECTED")
                print("   (This model has only 1 shallow tree. It cannot learn anything.)")
            else:
                print("\n✅ VERDICT: Looks like a trained model")
                
            # 4. Show the features it expects
            if hasattr(model, 'get_booster'):
                try:
                    features = model.get_booster().feature_names
                    print(f"📋 Features expected: {features}")
                except:
                    print("📋 Features: Could not read feature names")

    except Exception as e:
        print(f"❌ Error reading file: {e}")

# Run inspection on your files
files_to_check = [
    'XGBoost_target_win_model.pkl',
    'XGBoost_target_podium_model.pkl',
    'XGBoost_target_top10_model.pkl'
]

for f in files_to_check:
    inspect_pickle(f)